In [ ]:
import geopandas as gpd

import datetime
import cafo_iowa.db.session as s
import cafo_iowa.db.models as m

from cafo_iowa.data.helpers.geo import get_clusters
from cafo_iowa.utils.visualize import simple_map
from cafo_iowa.utils.utils import generate_random_points

In [ ]:
session = s.get_session()
engine = session.get_bind()

In [ ]:
barns = gpd.read_postgis(
    f"""
    WITH parcel_annotations AS (
        SELECT
            id,
            barn_cluster_id,
            parcel_id
        FROM
            processed.{m.BarnClusterParcels.__tablename__}
    )
    SELECT 
        id, 
        barn_cluster_id,
        geometry
    FROM 
        processed.{m.Barns.__tablename__} 
    WHERE 
        barn_cluster_id NOT IN (SELECT distinct barn_cluster_id FROM parcel_annotations)
    """,
    engine,
    geom_col="geometry",
)

In [ ]:
id_list = barns.groupby("barn_cluster_id")["id"].apply(list).reset_index()
clusters = barns.dissolve(by="barn_cluster_id").reset_index()
clusters = clusters.to_crs(4326)
clusters["lat"] = clusters.centroid.y
clusters["lon"] = clusters.centroid.x


clusters = clusters[["barn_cluster_id", "lat", "lon", "geometry"]]
clusters = clusters.merge(id_list, on="barn_cluster_id")

In [ ]:
print("Barns with no parcels:", clusters.id.explode().nunique())
print("Barn clusters with no parcels:", len(clusters))

In [ ]:
# get centroid of each cluster
clusters["centroid"] = clusters.geometry.centroid

# jitter points
new_locations = []
for _, row in clusters.iterrows():
    new_locations.extend(
        generate_random_points(
            row.barn_cluster_id, row.centroid, num_points=10, radius=50
        )
    )

# Create new GeoDataFrame
new_locations_gdf = gpd.GeoDataFrame(new_locations, crs=clusters.crs)

# Convert to lat/lon (EPSG:4326) for reGrid
new_locations_gdf = new_locations_gdf.to_crs(epsg=4326)
new_locations_gdf["lat"] = new_locations_gdf.geometry.y
new_locations_gdf["lon"] = new_locations_gdf.geometry.x

In [ ]:
# Save to CSV
date_today = datetime.datetime.now().strftime("%Y-%m-%d")

new_locations_gdf.to_csv(
    f"data/temp/{date_today}_barn_cluster_centroids_wo_parcels_20m_jitter.csv",
    index=False,
)